# 🍲 Indian Recipe Bot — Practical RAG Workshop

### From a messy CSV → a grounded, conversational AI system

**Stack:** Python + LangChain + Google Gemini

**What we're building:** a chatbot that takes the ingredients you already have and tells you what Gujarati or Punjabi dish you can actually cook tonight — grounded in a real recipe corpus, so it can't invent dishes that don't exist.

> 👋 **Not a Python expert? That's fine.** Every code cell is short, and the *what* and *why* are explained before and after it. The concepts — frameworks, memory, embeddings, retrieval, grounding — transfer to any language your team builds in later.

### Today's roadmap

1. Introduction to AI Frameworks
2. Configure the LLM (Gemini via LangChain)
3. LLM Parameters (temperature, max tokens, model choice)
4. Build a Basic Chatbot
5. Conversation History & Memory
6. Introduce RAG (Retrieval-Augmented Generation)
7. Build a Small RAG System
8. Final Mini AI System — the complete recipe bot

**How to work through this:** run each cell in order (Shift+Enter). Read the markdown *before* a cell (what we're about to do) and *after* it (what happened, and why). You need a free Gemini API key and nothing else.

---

## 0. Setup

Two things before we start: install the libraries, and load the API key.

**Getting a key:** go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey), sign in, click **Create API key**, copy it. The free tier is plenty for this notebook.

**Storing it:** create a file called `.env` next to this notebook containing one line:

```
GOOGLE_API_KEY=AIza...your-key...
```

`.env` is in `.gitignore`, so the key never reaches GitHub. The cell below falls back to a hidden prompt if it can't find the file — so this notebook works for anyone who clones the repo, without your key travelling with it.

In [ ]:
%pip install -q langchain langchain-core langchain-google-genai langchain-community pandas python-dotenv

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv()  # reads .env if it exists

if not os.environ.get("GOOGLE_API_KEY"):
    # No .env file - ask once, and keep it only in memory for this session.
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

---

## 1. Introduction to AI Frameworks

You could call the Gemini REST API directly with `requests`. For one prompt, that's simpler. The moment you want a **conversation**, or to **feed documents into the prompt**, or to **swap Gemini for another model**, you end up writing the same plumbing everyone else wrote.

LangChain is that plumbing, already written. Three ideas carry most of the value:

| Concept | What it is | Why you care |
|---|---|---|
| **Model wrapper** | One interface over Gemini, Claude, GPT, local models | Swap providers by changing one line |
| **Messages** | Typed `System` / `Human` / `AI` objects | Conversation history becomes a list, not string-concatenation |
| **Chains** | Piping components together with `\|` | Retrieve → format → generate reads as one expression |

The alternative isn't "no framework", it's "your own framework, written badly under time pressure". That said — a framework you don't understand is worse than none, which is why the next cell calls Gemini in about three lines and prints the raw result.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

response = llm.invoke("In one sentence: what is dhokla?")
print(response.content)

**What happened:** `ChatGoogleGenerativeAI` wrapped the Gemini API. `.invoke()` sent your string and returned an `AIMessage` object — `.content` is the text; the object also carries token counts and metadata.

Note what we did *not* write: no HTTP request, no JSON parsing, no auth header. The wrapper read `GOOGLE_API_KEY` from the environment by itself.

---

## 2. Configure the LLM

`gemini-2.0-flash` is the fast, cheap model — right for a chatbot that answers in a second. Gemini also offers `gemini-2.0-pro` (slower, better reasoning) and Google periodically ships new versions.

Beyond the model name, two settings matter more than everything else combined.

In [ ]:
# Same model, two very different personalities.

factual = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.0)
creative = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=1.0)

prompt = "Suggest a name for a Gujarati snack food stall."

print("temperature=0.0 →", factual.invoke(prompt).content)
print()
print("temperature=1.0 →", creative.invoke(prompt).content)

**What happened:** same model, same prompt, different answers.

**Temperature** controls how much randomness is allowed when picking the next token. At `0.0` the model takes the most likely token every time — repeatable, conservative, occasionally dull. At `1.0` it samples more freely — varied, surprising, and more likely to state something untrue.

Run that cell twice. The `temperature=0.0` line gives you the same answer both times; the `temperature=1.0` line probably doesn't.

**For our recipe bot we want temperature near 0.** The bot is reporting facts from a corpus — which recipe, which ingredients, how many you're missing. Creativity there is indistinguishable from making things up.

---

## 3. LLM Parameters

Three more settings worth knowing.

In [ ]:
chatty = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.2,
    max_output_tokens=60,     # hard ceiling on reply length
    top_p=0.95,               # nucleus sampling: consider tokens covering 95% of probability
)

reply = chatty.invoke("Explain what makes Gujarati food distinctive.")
print(reply.content)
print()
print("--- usage ---")
print(reply.usage_metadata)

**What happened:** the reply stopped early — `max_output_tokens=60` cut it off mid-thought. That's the parameter doing exactly its job, and it's worth seeing the failure mode once so you recognise it later: a truncated reply is a budget problem, not a model problem.

`usage_metadata` shows input and output token counts. Tokens are how you're billed and how context limits are measured — roughly ¾ of a word each in English, more for Gujarati or Hindi text.

| Parameter | What it does | Sensible default |
|---|---|---|
| `temperature` | Randomness of token choice | `0.0–0.3` for factual work |
| `max_output_tokens` | Hard cap on reply length | Set it — runaway replies cost money |
| `top_p` | Restricts sampling to the most probable tokens | `0.95`, rarely needs tuning |

Tune `temperature` first. `top_p` is the one people fiddle with to avoid thinking about `temperature`.

---

## 4. Build a Basic Chatbot

A chatbot is a model plus a **system prompt** — a standing instruction that shapes every reply without the user seeing it.

LangChain represents a conversation as a list of typed messages: `SystemMessage` (the instruction), `HumanMessage` (the user), `AIMessage` (the model). That typing matters later, when we start assembling history programmatically.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = (
    "You are a friendly Indian cooking assistant specialising in Gujarati "
    "and Punjabi vegetarian food. Keep answers to two or three sentences. "
    "Never suggest meat, fish or egg."
)

bot = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="What can I make with besan and curd?"),
]

print(bot.invoke(messages).content)

**What happened:** the system prompt steered the reply without appearing in it. Ask the same question with no system message and you'll get a longer, more generic answer that might well suggest an omelette.

**But notice what this bot is doing:** answering from what Gemini absorbed during training. It has never seen our recipe corpus. It's guessing — plausibly, because Gujarati cooking is well represented on the internet, but guessing. It cannot tell you whether a dish is in *our* 910 recipes, and it will happily invent one.

That gap is the entire reason RAG exists. Hold the thought.

---

## 5. Conversation History & Memory

Here's the thing that surprises people: **the model has no memory.** Every `.invoke()` is a blank slate. It feels conversational only because we resend the whole history each time.

In [ ]:
# Proof: ask a follow-up with no history.
forgetful = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="And how long does that take to cook?"),
]
print("WITHOUT history:")
print(bot.invoke(forgetful).content)

**What happened:** the model had no idea what "that" referred to, and either asked for clarification or guessed at a dish. Nothing is broken — the information genuinely was not in the request.

Now the same follow-up, with the earlier turns included.

In [ ]:
conversation = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="What can I make with besan and curd?"),
]
first_reply = bot.invoke(conversation)
conversation.append(first_reply)                       # the AI's turn joins the list

conversation.append(HumanMessage(content="And how long does that take to cook?"))
second_reply = bot.invoke(conversation)

print("WITH history:")
print(second_reply.content)
print()
print("messages now in the conversation:", len(conversation))

**What happened:** by appending the AI's reply and then the follow-up, "that" resolved correctly.

**This is what memory actually is** — a list you maintain and resend. There is no hidden state on Google's servers. Three consequences worth internalising:

1. **Cost grows with conversation length.** Every turn resends everything before it. A 50-turn chat pays for the first message 50 times.
2. **There's a ceiling.** Exceed the context window and the request fails.
3. **You control what's remembered.** Long conversations get trimmed to the last N turns, or summarised into a paragraph. That's a design decision, not a framework feature.

Our recipe bot keeps something smaller and sharper than raw history: a **pantry set**. "Add ginger" mutates a Python set. That's cheaper and more reliable than hoping the model correctly tracks your fridge across fifteen turns.

---

## 6. Introduce RAG

Back to the gap from section 4. Our chatbot answers from training data, which means it can:

- invent a dish that doesn't exist
- describe a real dish with the wrong ingredients
- never tell you whether something is in *your* collection

**Retrieval-Augmented Generation** fixes this by changing what goes into the prompt. Instead of *"answer this question"*, we send *"here are three real recipes; answer using only these"*.

```
Question → find relevant documents → put them in the prompt → model answers from them
   \_______________ retrieval ______________/   \____ augmented generation ____/
```

The model stops being a knowledge source and becomes a **language** source. It's good at phrasing; our corpus is the authority on facts.

**How does "find relevant documents" work?** Keyword search fails immediately — someone typing "aubergine" should match a recipe listing "brinjal". So we use **embeddings**: each text becomes a vector of numbers positioned so that similar meanings land near each other. Finding relevant documents becomes finding nearby vectors.

Let's look at real numbers.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector = embeddings.embed_query("brinjal curry")
print("dimensions:", len(vector))
print("first 8 numbers:", [round(v, 4) for v in vector[:8]])

**What happened:** a short phrase became a list of 768 numbers. No single number means anything on its own — meaning lives in the *direction* the whole vector points.

Now the part that makes retrieval work.

In [ ]:
import numpy as np

def similarity(a: str, b: str) -> float:
    """Cosine similarity between two texts: 1.0 = identical direction."""
    va, vb = embeddings.embed_query(a), embeddings.embed_query(b)
    va, vb = np.array(va), np.array(vb)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

pairs = [
    ("brinjal curry",  "aubergine curry"),   # same thing, different word
    ("brinjal curry",  "baingan bharta"),    # same vegetable, different language
    ("brinjal curry",  "chocolate cake"),    # unrelated
]
for a, b in pairs:
    print(f"{similarity(a, b):.3f}   {a:<16} vs {b}")

**What happened — and this is the payoff:** "brinjal" and "aubergine" score high despite sharing no letters. "Baingan bharta" scores high too, across languages. "Chocolate cake" scores low.

Keyword search would have scored all three pairs at zero overlap. Embeddings capture *meaning*, which is why they underpin every RAG system.

Two caveats worth carrying:

- **Embeddings are not understanding.** They capture statistical co-occurrence. Sometimes two things that merely appear in similar contexts score as similar.
- **Similarity is not relevance.** We'll hit this hard in section 8 — it's the most important idea in the notebook.

---

## 7. Build a Small RAG System

Time to use the real corpus. Four steps: **load → embed → retrieve → generate**.

### The data

`data/recipes_core.csv` holds 910 Indian vegetarian, eggless recipes, curated from the [Archana's Kitchen dataset](https://www.kaggle.com/datasets/kanishk307/6000-indian-food-recipes-dataset) of 6,871.

Getting from 6,871 to 910 was most of the real work, and the write-up in the README covers three findings worth a look — the dataset's `Diet` column labels *Singapore Style Chicken Fried Rice* as vegetarian, 427 rows spell it `Non Vegeterian`, and every Gujarati row carries an invisible byte-order mark that makes `cuisine == "Gujarati Recipes"` match nothing at all.

A fourth flaw only showed up when the ingredient parser was tested: **719 rows have Devanagari text sitting in the `Translated...` columns** — the translation step never ran for them. Some have English ingredients but Hindi instructions, so they parse fine and then hand the user cooking steps they may not read. They are dropped at build time.

That one is worth dwelling on. It was invisible in every row count and every spot-check; it surfaced as *12% of recipes parsing to zero ingredients*, which looked like a parser bug. Chasing the symptom found a data problem.

**The lesson that generalises: RAG quality is corpus quality.** A perfect retrieval pipeline over a dirty corpus returns dirt, confidently. Most of the work in a real RAG project is here, and none of it is visible in the demo.

In [ ]:
import pandas as pd

recipes = pd.read_csv("../data/recipes_core.csv")

print(f"{len(recipes)} recipes")
print(recipes.region.value_counts().to_string())
print()
recipes[["name", "region", "course"]].head(5)

### Turning rows into documents

A vector store holds **documents**: a chunk of text plus metadata. We build one string per recipe containing the parts worth matching on, and keep the rest as metadata to display later.

Note we're chunking *per recipe*, not splitting long text into fixed-size pieces. Chunking strategy is usually the biggest quality lever in RAG, and here the data hands us a natural unit — one recipe, one document. When your source is a 200-page PDF you have to invent that boundary, and choosing badly is the most common way RAG systems end up mediocre.

In [ ]:
from langchain_core.documents import Document

def to_document(row) -> Document:
    text = (
        f"{row['name']}\n"
        f"Cuisine: {row['cuisine']} ({row['region']})\n"
        f"Course: {row['course']}\n"
        f"Ingredients: {row['ingredients']}"
    )
    return Document(
        page_content=text,
        metadata={
            "name": row["name"],
            "region": row["region"],
            "course": row["course"],
            "time_mins": row["total_time_mins"],
            "url": row["url"],
            "ingredients": row["ingredients"],
        },
    )

documents = [to_document(r) for _, r in recipes.iterrows()]
print(f"{len(documents)} documents")
print(documents[0].page_content[:280])

### Embedding and storing

`InMemoryVectorStore` keeps the vectors in a Python list and compares them with brute force. For 910 documents that is a few milliseconds — genuinely faster than a database, with one fewer dependency.

You would reach for FAISS, Chroma or Pinecone somewhere north of ~100,000 documents, or when the index must outlive the process. Reaching for one at 910 documents and calling it architecture is a thing reviewers notice.

**This cell makes 910 API calls in a batch.** Give it a minute.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})

print("indexed:", len(documents), "recipes")

In [ ]:
# Retrieval on its own - no LLM involved yet.
for doc in retriever.invoke("something steamed with gram flour"):
    print("-", doc.metadata["name"], "|", doc.metadata["region"])

**What happened:** you asked in plain English, with no dish name, and got back steamed besan dishes. The retriever compared your question's vector against all 910 and returned the three nearest.

Nothing has been generated yet. This is pure search — and it's the half of RAG that determines whether the answers are any good.

### Adding generation

Now we hand those documents to Gemini with an instruction to answer **only** from them. The `|` operator pipes components together into a chain.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Gujarati and Punjabi vegetarian cooking assistant.\n"
     "Answer ONLY from the recipes provided below. Never invent a recipe, "
     "an ingredient or a cooking time. If the recipes do not answer the "
     "question, say so plainly.\n\n"
     "RECIPES:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs) -> str:
    return "\n\n---\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.0)
    | StrOutputParser()
)

print(rag_chain.invoke("What can I make that's steamed and uses gram flour?"))

**What happened:** read that chain right to left and it's four steps — the question fans out to the retriever *and* passes through unchanged, both land in the prompt template, the prompt goes to Gemini, and the output parser pulls out the text.

The reply is now grounded in real recipes from our corpus rather than Gemini's training data.

### Testing the grounding

A RAG system that answers everything confidently hasn't been tested. Ask it something the corpus cannot answer.

In [ ]:
print(rag_chain.invoke("How do I make chicken biryani?"))
print()
print("---")
print()
print(rag_chain.invoke("What is the capital of France?"))

**What happened:** the bot declined both. It has no chicken recipes and no geography, and the system prompt told it to say so rather than improvise.

**This is the single most valuable thing to demonstrate about a RAG system.** Anyone can show a bot answering a question well. Showing it *refuse* proves the grounding is real — that the model is constrained by the corpus rather than decorating it.

Gemini certainly knows the capital of France. It didn't say, because the instruction said not to.

---

## 8. Final Mini AI System — the complete recipe bot

Sections 1–7 gave us a working RAG pipeline. Now we make it a *product*, and this is where the interesting problem lives.

### Why plain RAG isn't enough here

Our retriever ranks by embedding similarity. For "what can I cook with what I have", similarity answers the wrong question.

**Cosine similarity is symmetric.** Given a pantry of *besan, curd, ginger, green chilli*, a sixteen-ingredient undhiyu that happens to contain all four scores beautifully — and you cannot cook it, because you're missing twelve things. A four-ingredient kadhi you *can* cook scores lower.

The user isn't asking *"which recipe is most similar to my ingredients?"* They're asking *"which recipe is most **covered** by my ingredients?"*

```
coverage = |pantry ∩ recipe| / |recipe|
```

Dividing by the **recipe's** size, not the pantry's, is the whole trick. It's asymmetric, and it punishes long recipes you can't complete.

So we keep both: retrieval for **recall** (tolerant of wording, handles vague questions), coverage for **precision** (orders by what's actually cookable). This is the standard *retrieve-then-rerank* pattern, and being able to say why it's two stages rather than one is worth more than any amount of weight-tuning.

In [ ]:
import re

def parse_ingredients(raw: str) -> set[str]:
    """Turn '2 tablespoon Gram flour (besan)' into {'gram flour', 'besan'}.

    The brackets are a gift: the dataset lists regional synonyms inside them,
    so 'Karela (Bitter Gourd/ Pavakkai)' hands us three names for one thing.
    """
    entities = set()
    for part in str(raw).split(","):
        part = part.strip().lower()
        if not part:
            continue
        # Bracketed synonyms become entities in their own right.
        for synonym in re.findall(r"\(([^)]*)\)", part):
            for piece in re.split(r"[/,]", synonym):
                if piece.strip():
                    entities.add(piece.strip())
        part = re.sub(r"\([^)]*\)", " ", part)      # drop the brackets
        part = re.sub(r"\s*-\s*.*$", "", part)       # drop '- finely chopped'
        # Drop leading quantity and unit: '2 tablespoon gram flour'
        part = re.sub(
            r"^[\d\s/¼½¾.]*\s*"
            r"(cups?|tablespoons?|teaspoons?|tbsp|tsp|grams?|gms?|kg|ml|litres?|"
            r"inch|cloves?|sprigs?|pinch|handfuls?|numbers?|nos?)?\s*",
            "", part)
        part = re.sub(r"[^a-z\s]", " ", part).strip()
        if len(part) > 2:
            entities.add(re.sub(r"\s+", " ", part))
    return entities

sample = recipes.iloc[0]
print(sample["name"])
print(sorted(parse_ingredients(sample["ingredients"]))[:12])

**What happened:** raw ingredient strings became a set of comparable entities, with the bracketed synonyms extracted as extra names. That gives us a free bilingual vocabulary — a user typing "besan" matches a recipe that says "gram flour".

This parser is rules-only and imperfect. It's also about thirty lines, and good enough that the coverage numbers mean something. Knowing when a rules-based approach is sufficient is a judgement worth having.

In [ ]:
# Pre-compute the ingredient set for every recipe, once.
recipes["entities"] = recipes["ingredients"].map(parse_ingredients)

W_COVERAGE, W_SIMILARITY, W_MISSING, W_REGION = 0.55, 0.25, 0.05, 0.15
REGION_PRIOR = {"Gujarati": 1.0, "Punjabi": 0.4}
FALLBACK_THRESHOLD = 0.34

def coverage(pantry: set[str], recipe_entities: set[str]) -> tuple[float, list[str]]:
    """Fraction of the recipe the user can supply, plus what's missing."""
    if not recipe_entities:
        return 0.0, []
    have = {e for e in recipe_entities if any(p in e or e in p for p in pantry)}
    missing = sorted(recipe_entities - have)
    return len(have) / len(recipe_entities), missing

# Demonstrate the asymmetry that makes this necessary.
pantry = {"besan", "curd", "ginger", "green chilli"}
for _, row in recipes.head(400).iterrows():
    cov, missing = coverage(pantry, row["entities"])
    if cov > 0.45:
        print(f"{cov:.0%}  missing {len(missing):>2}  {row['name'][:52]}")

**What happened:** recipes scored by how *completable* they are, not how similar. The percentage is the fraction of the dish you already have; the missing count is what you'd need to buy.

Notice that a recipe can share four ingredients with your pantry and still score badly — because it needs twenty more. That's the whole point, and it's invisible to embedding similarity.

### Putting it together

The full bot: retrieve broadly, re-rank by coverage, apply the Gujarati preference, fall back to the wider corpus when nothing in Gujarati or Punjabi is cookable — and say so when it does.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class RecipeBot:
    """Retrieval + coverage re-ranking + grounded generation."""

    pantry: set[str] = field(default_factory=set)

    def update_pantry(self, message: str) -> None:
        """Pantry lives in Python, not in the model's memory - cheaper and exact."""
        text = message.lower()
        if any(w in text for w in ("start over", "reset", "clear")):
            self.pantry.clear()
            return
        found = {e for e in re.split(r"[,.]| and ", text) if (e := e.strip())}
        remove = text.startswith(("remove", "no ", "without", "actually no"))
        for item in found:
            item = re.sub(r"^(add|remove|no|without|i have|actually no)\s+", "", item).strip()
            if 2 < len(item) < 30:
                self.pantry.discard(item) if remove else self.pantry.add(item)

    def rank(self, question: str, top_k: int = 4) -> tuple[list, bool]:
        """Score every recipe; return the best, and whether we had to widen."""
        semantic = {d.metadata["name"] for d in retriever.invoke(question)}
        scored = []
        for _, row in recipes.iterrows():
            cov, missing = coverage(self.pantry, row["entities"])
            score = (
                W_COVERAGE * cov
                + W_SIMILARITY * (1.0 if row["name"] in semantic else 0.0)
                - W_MISSING * min(len(missing), 10) / 10
                + W_REGION * REGION_PRIOR.get(row["region"], 0.0)
            )
            scored.append((score, cov, missing, row))
        scored.sort(key=lambda t: -t[0])
        widened = scored[0][1] < FALLBACK_THRESHOLD
        return scored[:top_k], widened

    def answer(self, message: str) -> str:
        self.update_pantry(message)
        if not self.pantry:
            return "Tell me what's in your kitchen and I'll find something you can cook."

        results, widened = self.rank(message)
        context = "\n\n---\n\n".join(
            f"{r['name']} ({r['region']}, {r['course']}, {r['total_time_mins']} mins)\n"
            f"You have {cov:.0%} of the ingredients. Missing: "
            f"{', '.join(missing[:6]) if missing else 'nothing'}\n"
            f"Ingredients: {r['ingredients'][:400]}"
            for _, cov, missing, r in results
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are a Gujarati and Punjabi vegetarian cooking assistant.\n"
             "Recommend from the candidates below and NOTHING else. Never invent "
             "a recipe or an ingredient. State the coverage percentage and what "
             "is missing for each. Prefer Gujarati dishes when coverage is "
             "comparable. Be concise: two sentences per recipe at most.\n"
             f"{'NOTE: nothing in the core Gujarati/Punjabi collection matched well. Say so clearly before recommending.' if widened else ''}\n\n"
             "CANDIDATES:\n{context}"),
            ("human", "My pantry: {pantry}\n\n{question}"),
        ])
        chain = prompt | ChatGoogleGenerativeAI(
            model="gemini-2.0-flash", temperature=0.1) | StrOutputParser()
        return chain.invoke({
            "context": context,
            "pantry": ", ".join(sorted(self.pantry)),
            "question": message,
        })

bot = RecipeBot()
print(bot.answer("I have besan, curd, ginger and green chilli"))

**What happened:** retrieval, coverage re-ranking, the Gujarati preference and grounded generation, in one call. Every number in the reply was computed by our code — Gemini phrased it, it didn't invent it.

### Multi-turn

In [ ]:
print("TURN 2 — adding to the pantry")
print(bot.answer("also add rice and jaggery"))
print()
print("pantry is now:", sorted(bot.pantry))

**What happened:** the pantry persisted and the recommendations shifted. Note it lives in a Python `set`, not in conversation history — exact, free to maintain, and impossible for the model to lose track of.

### The honest test

Every demo shows the happy path. Here's the one that matters.

In [ ]:
edge = RecipeBot()
print(edge.answer("I have broccoli, olives and feta cheese"))

**What happened:** nothing Gujarati or Punjabi is cookable from that pantry, so the bot said so instead of pretending. If it widened to the fallback tier, it announced that it was doing so.

A bot that never says "I don't have a good answer" isn't confident — it's untested.

---

## What we built

```
CSV (6,871) ──filter──▶ 910 curated recipes ──embed──▶ vector store
                                                          │
user message ──parse pantry──▶ semantic retrieval ────────┘
                                     │
                          coverage re-ranking + Gujarati prior
                                     │
                          Gemini, grounded in candidates ──▶ answer
```

### The five ideas worth keeping

1. **Memory is a list you resend.** There is no server-side conversation state.
2. **Embeddings capture meaning, not spelling.** "Brinjal" and "aubergine" are close; that's why retrieval beats keyword search.
3. **RAG changes the prompt, not the model.** You're not teaching Gemini anything — you're putting the right documents in front of it.
4. **Similarity is not relevance.** The most transferable idea here. Cosine similarity answered a question nobody asked; defining the right objective mattered more than any model choice.
5. **Corpus quality caps system quality.** A perfect pipeline over a dirty corpus returns dirt, confidently. Most of the real work was the filtering nobody sees.

### Where to take it next

- **Evaluation** — hand-written test queries, a baseline, and a metric. A result without a baseline isn't a result.
- **Streaming** — `.stream()` instead of `.invoke()` so replies appear as they're written.
- **A real interface** — the same engine behind Streamlit or FastAPI. Nothing above imports a UI framework, so that's additive.
- **Hybrid retrieval** — BM25 alongside embeddings, for exact dish names.